# Autoencoders (vanilla / denoising)

**Domain:** Architectures  ·  **recommended addition**  ·  **runnable:** yes

A refresher on the **autoencoder** — the simplest neural net that learns a
*compressed* representation by reconstructing its own input through a narrow
bottleneck. We cover the **vanilla** AE and its most useful sibling, the
**denoising** AE, and build both from scratch in NumPy on 8×8 digit images.

## 1. What & Why

An **autoencoder (AE)** is a network trained to copy its input to its output
through a **bottleneck** that is too small to memorize the data. Forcing the
signal through `code = encode(x)` and back out via `x̂ = decode(code)` makes the
bottleneck learn the *structure* of the data — the directions that actually vary —
rather than the raw pixels.

There are no labels: the input **is** the target. That makes the AE the canonical
**self-supervised / unsupervised** representation learner.

**The problem it solves.**

- **Dimensionality reduction** — a nonlinear cousin of PCA. The code is a dense,
  low-dimensional embedding you can cluster, visualize, or feed to a downstream model.
- **Denoising / inpainting** — train on corrupted→clean pairs and the AE learns to
  project noisy inputs back onto the data manifold.
- **Anomaly detection** — train only on "normal" data; inputs that reconstruct
  *poorly* (high error) are out-of-distribution.
- **Pretraining** — learn features without labels, then fine-tune the encoder.

**When *not* to reach for one.** A plain AE is **not a generative model** — its
latent space has holes, so decoding a random code gives garbage. If you want to
*sample* new data, use a **[VAE](vae.ipynb)** (probabilistic bottleneck),
a **[GAN](gan.ipynb)**, or a **[diffusion model](diffusion-models.ipynb)**. If your
data is roughly linear, **PCA** is faster and has a closed-form solution.

## 2. Mental Model

An autoencoder is an **hourglass**. Information is squeezed through a narrow waist
and must be reconstructed from whatever survives the squeeze:

```
        encoder                bottleneck              decoder
   x  ──────────▶  ░░░░░  ──▶  ● code (dim ≪ input)  ──▶  ░░░░░  ──▶  x̂
 (64 px)            wide        the compressed essence       wide      (64 px)
                              \________ reconstruct ________/
                                   loss = ‖ x − x̂ ‖²
```

The bottleneck is a **lossy compressor the network designs for itself**. Because
the code can't hold everything, gradient descent spends its capacity on whatever
reduces reconstruction error the most — the dominant, repeated structure of the data.

Two knobs change everything:

- **Squeeze the waist** (small code) → stronger compression, more abstraction.
- **Corrupt the input but keep the clean target** (denoising AE) → the net can't
  just copy pixels; it must learn what a *clean* example looks like.

## 3. Key Concepts

**Encoder / decoder.** `code = f(x)` and `x̂ = g(code)`. Usually MLPs or CNNs;
often roughly mirror-image in shape. The encoder is the part you keep for downstream
tasks.

**Bottleneck (latent code).** The low-dim layer in the middle. Its width is the
single most important hyperparameter — it sets how much can pass through.

**Undercomplete vs overcomplete.** *Undercomplete* = code smaller than input (the
usual case; the bottleneck does the regularizing). *Overcomplete* = code ≥ input,
which can trivially learn the identity unless you add a constraint (noise, sparsity,
dropout, weight tying).

**Reconstruction loss.** MSE `‖x − x̂‖²` for continuous data; binary cross-entropy
for pixels in `[0, 1]`.

**Linear AE ≈ PCA.** An AE with linear activations and MSE loss learns the same
subspace as PCA — it spans the top-`k` principal components (though not necessarily
orthonormal or ordered). Nonlinear activations are what let it beat PCA. We verify
this below.

**Denoising AE (DAE).** Corrupt the input `x̃ = x + noise`, feed `x̃`, but score
against the **clean** `x`. The objective becomes "learn to undo the corruption,"
which pushes the model to capture the data manifold instead of the identity.
(Vincent et al., 2008.)

**Regularized variants.** *Sparse* AE (penalize code activations), *contractive* AE
(penalize encoder Jacobian), *masked* AE (reconstruct hidden patches — the idea
behind MAE pretraining). All share the same "reconstruct under a constraint" recipe.

## 4. Setup

Pure **NumPy + scikit-learn** — no GPU, no deep-learning framework. We use
scikit-learn's `load_digits` (1797 hand-written digits, 8×8 = 64 pixels each),
which is tiny and ships with the library. In real projects you'd write the same
model in PyTorch/Keras in a few lines, but doing it by hand here makes the moving
parts unmistakable.

In [ ]:
# %pip install numpy scikit-learn   # both are usually already present
import numpy as np
from sklearn.datasets import load_digits

X = load_digits().data / 16.0          # (1797, 64), pixel values scaled to [0, 1]
N, D = X.shape
print(f"{N} images, {D} pixels each, values in [{X.min():.0f}, {X.max():.0f}]")

def show(vec, label=""):
    """Render a 64-vector as an 8x8 ASCII image."""
    chars = " .:-=+*#%@"
    grid = (vec.reshape(8, 8) * 9).round().astype(int).clip(0, 9)
    art = "\n".join("".join(chars[p] for p in row) for row in grid)
    print((label + "\n" if label else "") + art)

show(X[0], "a sample digit:")

## 5. Worked Examples

### Example 1 — A vanilla undercomplete autoencoder

One hidden bottleneck of **16 units** (a 4× squeeze from 64 pixels), `tanh`
encoder, `sigmoid` decoder, MSE loss, trained with plain full-batch gradient
descent. We backprop by hand so every gradient is visible.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

class AutoEncoder:
    """64 -> H -> 64, tanh encoder + sigmoid decoder, trained with MSE."""
    def __init__(self, d=D, h=16, seed=0):
        r = np.random.default_rng(seed)
        self.W1 = r.normal(0, 0.1, (d, h)); self.b1 = np.zeros(h)
        self.W2 = r.normal(0, 0.1, (h, d)); self.b2 = np.zeros(d)

    def encode(self, x):
        return np.tanh(x @ self.W1 + self.b1)

    def forward(self, x):
        h = self.encode(x)
        return h, sigmoid(h @ self.W2 + self.b2)

    def fit(self, X, target=None, iters=600, lr=0.5, log_every=150):
        target = X if target is None else target   # DAE passes a clean target
        n = len(X)
        for it in range(1, iters + 1):
            h, xhat = self.forward(X)
            diff = (xhat - target) / n              # dMSE/dxhat
            dz2 = diff * xhat * (1 - xhat)          # through sigmoid
            dW2 = h.T @ dz2;  db2 = dz2.sum(0)
            dz1 = (dz2 @ self.W2.T) * (1 - h ** 2)  # through tanh
            dW1 = X.T @ dz1;  db1 = dz1.sum(0)
            self.W2 -= lr * dW2; self.b2 -= lr * db2
            self.W1 -= lr * dW1; self.b1 -= lr * db1
            if log_every and it % log_every == 0:
                print(f"  iter {it:4d}   MSE {np.mean((xhat - target) ** 2):.4f}")
        return self

ae = AutoEncoder(h=16).fit(X)
_, recon = ae.forward(X)
print(f"\nfinal reconstruction MSE: {np.mean((recon - X) ** 2):.4f}")

In [ ]:
# Reconstruction is blurry but clearly recognizable — the 16-d code kept the shape.
i = 7
show(X[i],      f"original (digit, 64-d):")
print()
show(recon[i],  f"reconstruction (from 16-d code):")
print(f"\ncode (16 numbers): {np.round(ae.encode(X[i]), 2)}")

### Example 2 — A denoising autoencoder

Same architecture, one change in training: feed a **corrupted** copy of each image
but score against the **clean** original. The model can no longer cheat by copying
pixels — it has to learn what a clean digit looks like. At test time it cleans up
noise it has never seen.

In [ ]:
class DenoisingAutoEncoder(AutoEncoder):
    def fit(self, X, iters=600, lr=0.5, noise=0.6, log_every=150):
        r = np.random.default_rng(1)
        n = len(X)
        for it in range(1, iters + 1):
            Xn = np.clip(X + noise * r.normal(0, 1, X.shape), 0, 1)  # fresh noise each step
            h, xhat = self.forward(Xn)
            diff = (xhat - X) / n                  # target is the CLEAN image
            dz2 = diff * xhat * (1 - xhat)
            dW2 = h.T @ dz2;  db2 = dz2.sum(0)
            dz1 = (dz2 @ self.W2.T) * (1 - h ** 2)
            dW1 = Xn.T @ dz1; db1 = dz1.sum(0)
            self.W2 -= lr * dW2; self.b2 -= lr * db2
            self.W1 -= lr * dW1; self.b1 -= lr * db1
            if log_every and it % log_every == 0:
                print(f"  iter {it:4d}   clean-MSE {np.mean((xhat - X) ** 2):.4f}")
        return self

dae = DenoisingAutoEncoder(h=16).fit(X, noise=0.6)

# Corrupt a held-out-style image and ask the DAE to clean it.
rng = np.random.default_rng(42)
clean = X[7]
noisy = np.clip(clean + 0.6 * rng.normal(0, 1, clean.shape), 0, 1)
_, cleaned = dae.forward(noisy[None]); cleaned = cleaned[0]

show(clean,   "CLEAN target:");   print()
show(noisy,   "NOISY input:");    print()
show(cleaned, "DENOISED output:")
print(f"\nMSE noisy-vs-clean: {np.mean((noisy - clean)**2):.3f}"
      f"   ->   denoised-vs-clean: {np.mean((cleaned - clean)**2):.3f}")

### Example 3 — Linear AE ≈ PCA (the sanity check)

Swap the nonlinearities for the identity and an undercomplete linear AE spans the
same subspace as PCA. We compare the reconstruction error of a `k`-component PCA
against the theoretical optimum, confirming the bottleneck is "doing PCA" when it's
linear — and that nonlinearity is what buys you more.

In [ ]:
from sklearn.decomposition import PCA

k = 16
pca = PCA(n_components=k).fit(X)
X_pca = pca.inverse_transform(pca.transform(X))     # project to k-d and back
mse_pca = np.mean((X_pca - X) ** 2)

print(f"PCA({k}) reconstruction MSE        : {mse_pca:.4f}")
print(f"nonlinear AE(16) reconstruction MSE: {np.mean((recon - X) ** 2):.4f}")
print(f"variance retained by PCA({k})      : {pca.explained_variance_ratio_.sum():.1%}")
print("\nLinear bottleneck == PCA subspace; the AE's tanh/sigmoid let it bend the\n"
      "manifold, which is why a tiny 16-unit AE lands in the same ballpark as PCA-16.")

## 6. Gotchas & Pitfalls

- **Overcomplete + no constraint = identity function.** If the code is as wide as
  (or wider than) the input and you add no regularizer, the AE learns to copy
  perfectly and learns *nothing*. Keep it undercomplete, or add noise / sparsity /
  dropout / tied weights.

- **An AE is not generative.** The latent space is **not** smooth or gap-free.
  Decoding a random or interpolated code usually yields garbage. Reach for a
  **[VAE](vae.ipynb)** or diffusion model if you need to *sample*.

- **Loss must match the data.** MSE assumes Gaussian-ish continuous targets; for
  pixels in `[0,1]` binary cross-entropy often trains faster and sharper. Mismatched
  loss → muddy reconstructions.

- **Blurry reconstructions are inherent.** MSE averages over plausible outputs, so
  AE/VAE images look soft. That's the loss, not a bug — adversarial or diffusion
  objectives are what produce crisp samples.

- **Forgetting to scale inputs.** A `sigmoid` decoder outputs `[0,1]`; if your data
  isn't in that range the loss can't reach zero. Normalize first (we divided by 16).

- **Too aggressive a bottleneck.** Squeeze too hard and even the structure won't
  fit — reconstructions collapse toward the dataset mean. Tune the code width.

- **Denoising: re-sample noise every step.** Corrupting once and reusing it lets the
  model memorize *that* corruption. Draw fresh noise per batch (we do, inside `fit`).

- **Anomaly detection drift.** Reconstruction-error thresholds assume the "normal"
  distribution is stationary. If it shifts, your threshold silently goes stale.

## 7. When to Use vs Alternatives

| Goal | Use an autoencoder? | Better alternative |
|------|---------------------|--------------------|
| Linear dimensionality reduction | Overkill | **PCA** — closed-form, fast, interpretable |
| Nonlinear embedding for downstream ML | ✅ Yes | UMAP/t-SNE for *visualization* only (no decoder) |
| Denoising / inpainting | ✅ Denoising AE | Diffusion models for SOTA quality |
| Anomaly / novelty detection | ✅ Reconstruction error | Isolation Forest, One-Class SVM for tabular |
| **Generating** new samples | ❌ No (holey latent) | **[VAE](vae.ipynb)**, **[GAN](gan.ipynb)**, **[diffusion](diffusion-models.ipynb)** |
| Self-supervised pretraining (vision) | ✅ Masked AE | Contrastive (SimCLR/DINO) when labels are scarce |
| Compression for storage/transmission | Rarely | Hand-tuned codecs (JPEG, etc.) usually win |

**Rule of thumb.** Use a plain AE when you want a **learned, nonlinear, reusable
encoder** and you'll only ever *encode/reconstruct* real inputs. The moment you need
to **sample**, **measure likelihood**, or **interpolate smoothly** in latent space,
graduate to a VAE — same hourglass, but the bottleneck becomes a *distribution*.

## 8. Resources

- **Deep Learning (Goodfellow, Bengio, Courville) — Ch. 14, "Autoencoders"** — the
  standard reference: undercomplete, sparse, denoising, contractive.
  <https://www.deeplearningbook.org/contents/autoencoders.html>
- **Vincent et al., "Extracting and Composing Robust Features with Denoising
  Autoencoders" (ICML 2008)** — the denoising-AE paper.
  <https://www.cs.toronto.edu/~larocheh/publications/icml-2008-denoising-autoencoders.pdf>
- **Keras tutorial, "Building Autoencoders in Keras"** — clean, runnable code for
  vanilla, sparse, denoising, and convolutional AEs.
  <https://blog.keras.io/building-autoencoders-in-keras.html>
- **He et al., "Masked Autoencoders Are Scalable Vision Learners" (CVPR 2022)** — how
  the reconstruct-under-a-mask idea became a top self-supervised pretraining recipe.
  <https://arxiv.org/abs/2111.06377>
- **Cross-links in this library:** [VAE](vae.ipynb) (probabilistic bottleneck),
  [U-Net](u-net.ipynb) (encoder–decoder with skip connections),
  [diffusion models](diffusion-models.ipynb) (generative denoising).